# Electric Line Extension - Analysis 5
#### Descriptive Data
<i>Any question regarding the notebook, please contact Robert Ford<br>
    Last Updated: 05/31/2026 | Start Development: 04/07/2026</i>
* Utility / IOU Data from PG&E, SDG&E, and SCE for Q1–Q4 2025
* Goals:
  1. **Step 1** – Clean the base quarterly files (from `processed/`)
  2. **Step 2** – Export cleaned quarterly files + cleaned Master file
  3. **Step 3** – Melt the Master into long format for Tableau analysis

**Output files (all written to `processed/`):**
- `Condensed_Q1_2025_Cleaned.xlsx`
- `Condensed_Q2_2025_Cleaned.xlsx`
- `Condensed_Q3_2025_Cleaned.xlsx`
- `Condensed_Q4_2025_Cleaned.xlsx`
- `Master_Q1-Q4_2025_Cleaned.xlsx`
- `IOU_2025_Final.xlsx`

In [17]:
# ============================================================
# CELL 1 – Import Libraries
# ============================================================
import pandas as pd
import numpy as np
import os
from pathlib import Path

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [18]:
# ============================================================
# CELL 2 – Folder Paths
# ============================================================
# Input files come from 'processed' (already condensed per-quarter).
# NEVER read from or write to 'raw'.

processed_folder = (
    r"C:/Users/Rford/OneDrive - California Energy Commission"
    r"/Documents/Analysis - Scripts and Code"
    r"/Electric-Line-Extension-Data/data/processed"
)

os.makedirs(processed_folder, exist_ok=True)

# Input quarterly files (the base files to clean)
input_files = {
    "Q1": os.path.join(processed_folder, "Condensed_Q1_2025.xlsx"),
    "Q2": os.path.join(processed_folder, "Condensed_Q2_2025.xlsx"),
    "Q3": os.path.join(processed_folder, "Condensed_Q3_2025.xlsx"),
    "Q4": os.path.join(processed_folder, "Condensed_Q4_2025.xlsx"),
}

print('Folder paths set.')
for q, p in input_files.items():
    exists = os.path.exists(p)
    print(f'  {q}: {p}  [{"FOUND" if exists else "NOT FOUND – check path"}'  + ']')

Folder paths set.
  Q1: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\Condensed_Q1_2025.xlsx  [NOT FOUND – check path]
  Q2: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\Condensed_Q2_2025.xlsx  [NOT FOUND – check path]
  Q3: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\Condensed_Q3_2025.xlsx  [NOT FOUND – check path]
  Q4: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\Condensed_Q4_2025.xlsx  [NOT FOUND – check path]


In [19]:
# ============================================================
# CELL 3 – Canonical Column Names
# These are the target column names after all renaming.
# ============================================================

CANONICAL_COLUMNS = [
    "Year",
    "IOU",
    "New Construction / Upgrade",
    "Mixed Fuel / All Electric",
    "Month",
    "Customer Class",
    "Baseline / Climate Zone",
    "Multi Dwelling",
    "Total Discounts (Non-Exempted Projects)",
    "Total Number of Discounts (Non-Exempted Projects)",
    "Total Allowances (Non-Exempted Projects)",
    "Total Number of Allowances (Non-Exempted Projects)",
    "Total Refund Payments Provided to Builders (Non-Exempted Projects)",
    "Total Number of Refund Payments Provided to Builders (Non-Exempted Projects)",
    "Total Discounts (Exempted projects)",
    "Total Number of Discounts (Exempted Projects)",
    "Total Allowances (Exempted projects)",
    "Total Number of Allowances (Exempted Projects)",
    "Total Refund Payments Provided to Builders (Exempted projects)",
    "Total Number of Refund Payments Provided to Builders (Exempted Projects)",
    "Total Estimated Non-Refundable",
    "Total Estimated Refundable",
    "Average of # of Days Between Full Payment and Project Energization",
    "Total Electric Line or Service Extensions Energized",
    "Total Electric Line or Service Extension Applications (IOU Installed)",
    "Total Electric Line or Service Extension Applications (Applicant Installed)",
    "Total Number of Estimated Non-Refundable",
    "Total Number of Estimated Refundable",
    "Average of # of Days Between Full Payment and Project Energization (Completions Only)",
    "Mixed-Fuel New Construction Actual Costs",
    "Total Actual Cost Billing",
]

print(f'{len(CANONICAL_COLUMNS)} canonical columns defined.')

31 canonical columns defined.


In [20]:
# ============================================================
# CELL 4 – Helper: Month Abbreviation Mapping
# ============================================================

MONTH_MAP = {
    # Full names
    'january': 'Jan', 'february': 'Feb', 'march': 'Mar',
    'april': 'Apr', 'may': 'May', 'june': 'Jun',
    'july': 'Jul', 'august': 'Aug', 'september': 'Sep',
    'october': 'Oct', 'november': 'Nov', 'december': 'Dec',
    # Numeric strings (in case stored as 1, 2, ... or '01', '02', ...)
    '1': 'Jan', '2': 'Feb', '3': 'Mar', '4': 'Apr',
    '5': 'May', '6': 'Jun', '7': 'Jul', '8': 'Aug',
    '9': 'Sep', '10': 'Oct', '11': 'Nov', '12': 'Dec',
    '01': 'Jan', '02': 'Feb', '03': 'Mar', '04': 'Apr',
    '05': 'May', '06': 'Jun', '07': 'Jul', '08': 'Aug',
    '09': 'Sep',
    # Already-abbreviated (normalise capitalisation)
    'jan': 'Jan', 'feb': 'Feb', 'mar': 'Mar', 'apr': 'Apr',
    'jun': 'Jun', 'jul': 'Jul', 'aug': 'Aug', 'sep': 'Sep',
    'oct': 'Oct', 'nov': 'Nov', 'dec': 'Dec',
}


def standardise_month(val):
    """Convert any month representation to 3-letter abbreviation."""
    if pd.isna(val):
        return val
    # Handle datetime / Timestamp objects
    if hasattr(val, 'strftime'):
        return val.strftime('%b')
    s = str(val).strip()
    # Handle date strings like '2025-01-01' or '01/01/2025'
    for fmt in ('%Y-%m-%d', '%m/%d/%Y', '%d/%m/%Y', '%Y-%m-%d %H:%M:%S'):
        try:
            return pd.to_datetime(s, format=fmt).strftime('%b')
        except (ValueError, TypeError):
            pass
    key = s.lower()[:3] if len(s) > 3 else s.lower()
    # Try full-name key first, then 3-char prefix
    full_key = s.lower()
    if full_key in MONTH_MAP:
        return MONTH_MAP[full_key]
    if key in MONTH_MAP:
        return MONTH_MAP[key]
    # Numeric
    if s in MONTH_MAP:
        return MONTH_MAP[s]
    return s  # return as-is if unknown


print('Month helper defined.')

Month helper defined.


In [21]:
# ============================================================
# CELL 5 – Main Cleaning Function (Step 1)
# All cleaning steps are applied in the correct dependency order.
# ============================================================

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies all Step-1 cleaning rules to a quarterly dataframe.
    Returns a cleaned copy; the original is not modified.
    """
    df = df.copy()

    # ----------------------------------------------------------
    # 1. Strip whitespace from column headers
    # ----------------------------------------------------------
    df.columns = [
        str(c).strip().replace('\n', ' ')
        for c in df.columns
    ]
    # Collapse internal multiple spaces
    df.columns = [' '.join(c.split()) for c in df.columns]

    # ----------------------------------------------------------
    # 2. Rename duplicate / legacy column names
    # Per the spec:
    #   "Total Electric Line Extension Applications (IOU Installed)"
    #       → "Total Electric Line or Service Extension Applications (IOU Installed)"
    #   "Total Electric Line Extension Applications (Applicant Installed)"
    #       → "Total Electric Line or Service Extension Applications (Applicant Installed)"
    # If BOTH the old and new names exist, sum/merge; otherwise just rename.
    # ----------------------------------------------------------
    rename_map = {
        'Total Electric Line Extension Applications (IOU Installed)':
            'Total Electric Line or Service Extension Applications (IOU Installed)',
        'Total Electric Line Extension Applications (Applicant Installed)':
            'Total Electric Line or Service Extension Applications (Applicant Installed)',
    }

    for old_name, new_name in rename_map.items():
        if old_name in df.columns:
            if new_name in df.columns:
                # Both exist – fill NaN in the canonical column with values from the legacy column
                df[new_name] = df[new_name].combine_first(df[old_name])
                df.drop(columns=[old_name], inplace=True)
            else:
                df.rename(columns={old_name: new_name}, inplace=True)

    # ----------------------------------------------------------
    # 3. Strip whitespace from all string cell values
    # ----------------------------------------------------------
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )

    # ----------------------------------------------------------
    # 4. Drop summary / grand-total rows
    #    Criterion: 'Customer Class' is empty/NaN
    # ----------------------------------------------------------
    if 'Customer Class' in df.columns:
        df = df[df['Customer Class'].notna()]
        df = df[df['Customer Class'].astype(str).str.strip() != '']
        df.reset_index(drop=True, inplace=True)

    # ----------------------------------------------------------
    # 5. Year – ensure numeric integer 2025
    # ----------------------------------------------------------
    if 'Year' in df.columns:
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce').fillna(2025).astype(int)

    # ----------------------------------------------------------
    # 6. New Construction / Upgrade – "Existing" → "Upgrade"
    # ----------------------------------------------------------
    if 'New Construction / Upgrade' in df.columns:
        df['New Construction / Upgrade'] = (
            df['New Construction / Upgrade']
            .astype(str)
            .str.strip()
            .replace({'Existing': 'Upgrade'})
        )

    # ----------------------------------------------------------
    # 7. Month – standardise to 3-letter abbreviation text
    # ----------------------------------------------------------
    if 'Month' in df.columns:
        df['Month'] = df['Month'].apply(standardise_month)

    # ----------------------------------------------------------
    # 8. Customer Class – standardise values
    # ----------------------------------------------------------
    if 'Customer Class' in df.columns:
        cc_map = {
            # Agricultural variants → Agriculture
            'Agricultural': 'Agriculture',
            # Agency
            'Agency (City,County)': 'Agency (City, County, Caltrans)',
            # Mixed Use variants
            'Mixed Use': 'Mixed Use (Commercial / Residential)',
            'Mixed Use (Residential / Commercial)': 'Mixed Use (Commercial / Residential)',
            # Street lighting
            'Street and Outdoor area Lighting': 'Street Outdoor Areas',
            'Street and Outdoor Area Lighting': 'Street Outdoor Areas',
            'Street And Outdoor Area Lighting': 'Street Outdoor Areas',
        }
        df['Customer Class'] = df['Customer Class'].replace(cc_map)

        # Catch any remaining Agricultural variants (e.g. 'Agricultural - Pumping')
        df['Customer Class'] = df['Customer Class'].apply(
            lambda x: 'Agriculture' if isinstance(x, str) and x.lower().startswith('agricultural') else x
        )

    # ----------------------------------------------------------
    # 9. Baseline / Climate Zone – blanks → "Unknown"
    # ----------------------------------------------------------
    if 'Baseline / Climate Zone' in df.columns:
        df['Baseline / Climate Zone'] = df['Baseline / Climate Zone'].apply(
            lambda x: 'Unknown' if (pd.isna(x) or str(x).strip() == '') else x
        )

    # ----------------------------------------------------------
    # 10. Multi Dwelling – standardise values
    #     Must come AFTER Customer Class cleaning so we can use
    #     the correct cleaned Customer Class values.
    # ----------------------------------------------------------
    if 'Multi Dwelling' in df.columns:
        md_map = {
            'Multi Family': 'Multifamily',
            'Multi-Family': 'Multifamily',
            'Single-Family': 'Single Family',
            'Not Available': 'Unknown',
        }
        df['Multi Dwelling'] = df['Multi Dwelling'].replace(md_map)

        # Residential Customer Classes that can have Multifamily / Single Family
        residential_classes = {'Residential', 'Mixed Use (Commercial / Residential)'}

        def clean_multi_dwelling(row):
            val = row.get('Multi Dwelling', None)
            cc  = row.get('Customer Class', '')
            is_empty = pd.isna(val) or str(val).strip() == ''
            if is_empty:
                if cc not in residential_classes:
                    return 'Nonresidential'
                return 'Unknown'
            return val

        df['Multi Dwelling'] = df.apply(clean_multi_dwelling, axis=1)

    # ----------------------------------------------------------
    # 11. Mixed Fuel / All Electric – "Mixed" → "Mixed Fuel"
    # ----------------------------------------------------------
    if 'Mixed Fuel / All Electric' in df.columns:
        df['Mixed Fuel / All Electric'] = (
            df['Mixed Fuel / All Electric']
            .astype(str)
            .str.strip()
            .replace({'Mixed': 'Mixed Fuel'})
        )

    return df


print('Cleaning function defined.')

Cleaning function defined.


In [22]:
# ============================================================
# CELL 6 – Step 1 & 2: Load, Clean, and Export Quarterly Files
# ============================================================

cleaned_quarters = {}  # will hold cleaned DataFrames for each quarter

for quarter, filepath in input_files.items():
    print(f'\n--- {quarter} ---')

    if not os.path.exists(filepath):
        print(f'  SKIPPED – file not found: {filepath}')
        continue

    # Load
    df_raw = pd.read_excel(filepath)
    print(f'  Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

    # Clean
    df_clean = clean_dataframe(df_raw)
    print(f'  After cleaning: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')

    # Tag quarter for reference
    cleaned_quarters[quarter] = df_clean

    # Export cleaned quarterly file
    out_path = os.path.join(processed_folder, f'Condensed_{quarter}_2025_Cleaned.xlsx')
    df_clean.to_excel(out_path, index=False)
    print(f'  Saved: {out_path}')

print('\n✔ All quarterly files cleaned and exported.')


--- Q1 ---
  SKIPPED – file not found: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\Condensed_Q1_2025.xlsx

--- Q2 ---
  SKIPPED – file not found: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\Condensed_Q2_2025.xlsx

--- Q3 ---
  SKIPPED – file not found: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\Condensed_Q3_2025.xlsx

--- Q4 ---
  SKIPPED – file not found: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\Condensed_Q4_2025.xlsx

✔ All quarterly files cleaned and exported.


In [23]:
# ============================================================
# CELL 7 – Step 2: Build and Export Cleaned Master File
# ============================================================

if cleaned_quarters:
    master_df = pd.concat(cleaned_quarters.values(), ignore_index=True)
    print(f'Master shape: {master_df.shape[0]:,} rows × {master_df.shape[1]} columns')

    master_path = os.path.join(processed_folder, 'Master_Q1-Q4_2025_Cleaned.xlsx')
    master_df.to_excel(master_path, index=False)
    print(f'Saved Master: {master_path}')
else:
    print('No cleaned quarters available – check input file paths above.')

No cleaned quarters available – check input file paths above.


In [24]:
# ============================================================
# CELL 8 – Step 3: Build Fuel_Structure_Type column
# Combines 'New Construction / Upgrade' + 'Mixed Fuel / All Electric'
# Logic:
#   - If New Construction/Upgrade == 'New Construction' AND Mixed == 'All Electric'
#       → 'All Electric New Construction'
#   - If New Construction/Upgrade == 'Upgrade'          AND Mixed == 'All Electric'
#       → 'All Electric Upgrade'
#   - If New Construction/Upgrade == 'New Construction' AND Mixed == 'Mixed Fuel'
#       → 'Mixed Fuel New Construction'
#   - If New Construction/Upgrade == 'Upgrade'          AND Mixed == 'Mixed Fuel'
#       → 'Mixed Fuel Upgrade'
#   - If New Construction/Upgrade == 'Other'            AND Mixed == 'All Electric'
#       → 'All Electric'   (per spec: 'Other' rows all correspond to All Electric)
#   - Any remaining combination → Mixed value only (fallback)
# ============================================================

def build_fuel_structure_type(row):
    nc  = str(row.get('New Construction / Upgrade', '')).strip()
    mix = str(row.get('Mixed Fuel / All Electric', '')).strip()

    if nc == 'Other':
        # Per spec, 'Other' always pairs with All Electric
        return 'All Electric'

    combo = f'{nc}|{mix}'
    mapping = {
        'New Construction|All Electric': 'All Electric New Construction',
        'Upgrade|All Electric':          'All Electric Upgrade',
        'New Construction|Mixed Fuel':   'Mixed Fuel New Construction',
        'Upgrade|Mixed Fuel':            'Mixed Fuel Upgrade',
    }
    return mapping.get(combo, mix)  # fallback: use Mixed Fuel / All Electric value


master_df['Fuel_Structure_Type'] = master_df.apply(build_fuel_structure_type, axis=1)

print('Fuel_Structure_Type value counts:')
print(master_df['Fuel_Structure_Type'].value_counts())

Fuel_Structure_Type value counts:
Fuel_Structure_Type
All Electric New Construction    3397
All Electric Upgrade             2140
Mixed Fuel New Construction      1546
Mixed Fuel Upgrade                747
All Electric                       40
Name: count, dtype: int64


In [25]:
# ============================================================
# CELL 9 – Step 3: Define ID columns and value (melt) columns
# ============================================================

# Columns that identify each record – NOT melted
ID_COLUMNS = [
    'Month',
    'Year',
    'Customer Class',
    'Fuel_Structure_Type',
    'IOU',
    'Baseline / Climate Zone',
    'Multi Dwelling',
]

# Financial / count columns to melt into Type + Count
# All numeric columns not in ID_COLUMNS are candidates.
# Explicitly exclude the two source columns that were merged into Fuel_Structure_Type.
EXCLUDE_FROM_MELT = set(ID_COLUMNS) | {
    'New Construction / Upgrade',
    'Mixed Fuel / All Electric',
}

VALUE_COLUMNS = [
    col for col in master_df.columns
    if col not in EXCLUDE_FROM_MELT
]

print(f'{len(VALUE_COLUMNS)} columns will be melted into Type/Count:')
for c in VALUE_COLUMNS:
    print(f'  {c}')

23 columns will be melted into Type/Count:
  Total Discounts (Non-Exempted Projects)
  Total Number of Discounts (Non-Exempted Projects)
  Total Allowances (Non-Exempted Projects)
  Total Number of Allowances (Non-Exempted Projects)
  Total Refund Payments Provided to Builders (Non-Exempted Projects)
  Total Number of Refund Payments Provided to Builders (Non-Exempted Projects)
  Total Discounts (Exempted projects)
  Total Number of Discounts (Exempted Projects)
  Total Allowances (Exempted projects)
  Total Number of Allowances (Exempted Projects)
  Total Refund Payments Provided to Builders (Exempted projects)
  Total Number of Refund Payments Provided to Builders (Exempted Projects)
  Total Estimated Non-Refundable
  Total Estimated Refundable
  Average of # of Days Between Full Payment and Project Energization
  Total Electric Line or Service Extensions Energized
  Total Electric Line or Service Extension Applications (IOU Installed)
  Total Electric Line or Service Extension Appli

In [26]:
# ============================================================
# CELL 10 – Step 3: Melt and Export Final File
# NaN strategy: fill with 0 and keep all rows so no records are
# silently lost. A diagnostic block shows how many NaNs were
# present per value column before filling.
# ============================================================

# Verify all ID columns exist in master_df
missing_id = [c for c in ID_COLUMNS if c not in master_df.columns]
if missing_id:
    print(f'WARNING – these ID columns are missing from master_df: {missing_id}')
    print('Melting will proceed with available columns.')
    id_cols_present = [c for c in ID_COLUMNS if c in master_df.columns]
else:
    id_cols_present = ID_COLUMNS

# --- NaN diagnostic (run before filling so you can see what was blank) ---
print('NaN counts per value column (before fill):')
nan_counts = master_df[VALUE_COLUMNS].isna().sum()
nan_counts_nonzero = nan_counts[nan_counts > 0]
if nan_counts_nonzero.empty:
    print('  None – no NaNs found in value columns.')
else:
    for col, cnt in nan_counts_nonzero.items():
        pct = cnt / len(master_df) * 100
        print(f'  {col}: {cnt:,} NaNs ({pct:.1f}% of rows)')
total_nans = nan_counts.sum()
print(f'\nTotal NaN cells across all value columns: {total_nans:,}')
print(f'These will be replaced with 0; no rows will be dropped.\n')

# --- Fill NaN with 0 in value columns before melting ---
master_for_melt = master_df.copy()
master_for_melt[VALUE_COLUMNS] = master_for_melt[VALUE_COLUMNS].fillna(0)

# --- Melt ---
melted_df = master_for_melt.melt(
    id_vars=id_cols_present,
    value_vars=VALUE_COLUMNS,
    var_name='Type',
    value_name='Count',
)

melted_df.reset_index(drop=True, inplace=True)

print(f'Melted shape: {melted_df.shape[0]:,} rows × {melted_df.shape[1]} columns')
print()
print('Column summary:')
print(melted_df.dtypes)

# Export
final_path = os.path.join(processed_folder, 'IOU_2025_Final.xlsx')
melted_df.to_excel(final_path, index=False)
print(f'\nSaved final melted file: {final_path}')

NaN counts per value column (before fill):
  Total Discounts (Non-Exempted Projects): 4,324 NaNs (54.9% of rows)
  Total Number of Discounts (Non-Exempted Projects): 4,623 NaNs (58.7% of rows)
  Total Allowances (Non-Exempted Projects): 4,324 NaNs (54.9% of rows)
  Total Number of Allowances (Non-Exempted Projects): 4,458 NaNs (56.6% of rows)
  Total Refund Payments Provided to Builders (Non-Exempted Projects): 5,655 NaNs (71.9% of rows)
  Total Number of Refund Payments Provided to Builders (Non-Exempted Projects): 5,655 NaNs (71.9% of rows)
  Total Discounts (Exempted projects): 4,276 NaNs (54.3% of rows)
  Total Number of Discounts (Exempted Projects): 4,276 NaNs (54.3% of rows)
  Total Allowances (Exempted projects): 4,276 NaNs (54.3% of rows)
  Total Number of Allowances (Exempted Projects): 4,276 NaNs (54.3% of rows)
  Total Refund Payments Provided to Builders (Exempted projects): 5,656 NaNs (71.9% of rows)
  Total Number of Refund Payments Provided to Builders (Exempted Project

C:\Users\Rford\AppData\Local\Temp\ipykernel_42024\1803044359.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  master_for_melt[VALUE_COLUMNS] = master_for_melt[VALUE_COLUMNS].fillna(0)



Saved final melted file: C:/Users/Rford/OneDrive - California Energy Commission/Documents/Analysis - Scripts and Code/Electric-Line-Extension-Data/data/processed\IOU_2025_Final.xlsx


In [ ]:
# ============================================================
# CELL 11 – Verification Summary
# Quick sanity checks on the melted output.
# ============================================================

print('=== Verification Summary ===')
print()

print('--- Month unique values ---')
print(sorted(melted_df['Month'].dropna().unique()))
print()

print('--- Year unique values ---')
print(sorted(melted_df['Year'].dropna().unique()))
print()

print('--- Customer Class unique values ---')
for v in sorted(melted_df['Customer Class'].dropna().unique()):
    print(f'  {v}')
print()

print('--- Fuel_Structure_Type unique values ---')
print(sorted(melted_df['Fuel_Structure_Type'].dropna().unique()))
print()

print('--- IOU unique values ---')
print(sorted(melted_df['IOU'].dropna().unique()))
print()

print('--- Multi Dwelling unique values ---')
print(sorted(melted_df['Multi Dwelling'].dropna().unique()))
print()

print('--- Type (melted) unique values ---')
for v in sorted(melted_df['Type'].dropna().unique()):
    print(f'  {v}')
print()

print('=== Done – all 6 output files created ===')
print()
outputs = [
    'Condensed_Q1_2025_Cleaned.xlsx',
    'Condensed_Q2_2025_Cleaned.xlsx',
    'Condensed_Q3_2025_Cleaned.xlsx',
    'Condensed_Q4_2025_Cleaned.xlsx',
    'Master_Q1-Q4_2025_Cleaned.xlsx',
    'IOU_2025_Final.xlsx',
]
for f in outputs:
    p = os.path.join(processed_folder, f)
    exists = os.path.exists(p)
    print(f'  {"✔" if exists else "✗"} {f}')

=== Verification Summary ===

--- Month unique values ---
['Apr', 'Aug', 'Dec', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Sep']

--- Year unique values ---
[np.int64(2025)]

--- Customer Class unique values ---
  Agency (City, County, Caltrans)
  Agriculture
  Commercial
  Industrial
  Mixed Use (Commercial / Residential)
  Residential
  Street Outdoor Areas
  Telecommunications

--- Fuel_Structure_Type unique values ---
['All Electric', 'All Electric New Construction', 'All Electric Upgrade', 'Mixed Fuel New Construction', 'Mixed Fuel Upgrade']

--- IOU unique values ---
['PG&E', 'SCE', 'SDG&E']

--- Multi Dwelling unique values ---
['Multifamily', 'Nonresidential', 'Single Family', 'Unknown']

--- Type (melted) unique values ---
  Average of # of Days Between Full Payment and Project Energization
  Average of # of Days Between Full Payment and Project Energization (Completions Only)
  Mixed-Fuel New Construction Actual Costs
  Total Actual Cost Billing
  Total Allowanc